## Cell 1 — Install dependencies (run once, kernel auto-restarts)

Detects bad numpy/scipy versions, fixes them, and restarts the kernel.
**You'll see "Your session crashed" — that's expected**, just wait for auto-reconnect.

In [2]:
import os, sys, subprocess, time

# Detect: does numpy need fixing?
need_install = True
try:
    import numpy
    if numpy.__version__.startswith("2"):
        need_install = False
except Exception:
    pass

if need_install:
    print("Fixing numpy/scipy compatibility...")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "boltz"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "numpy>=2.0", "scipy>=1.13"], check=True)
    print("✓ Done. Restarting kernel in 2s — re-run Cell 2 after restart.")
    time.sleep(2)
    os.kill(os.getpid(), 9)
else:
    print("✓ Numpy already ≥ 2.0 — skip to Cell 2.")

✓ Numpy already ≥ 2.0 — skip to Cell 2.


## Cell 2 — Run the BindCraft pipeline

Installs BindCraft (~6 min) on first run, then runs the design pipeline (~3.5 h for 20 designs).

**Keep this cell running.** Free Colab disconnects idle browsers after ~90 min — keep the tab visible. If it disconnects, BindCraft resumes from where it left off when you re-run (outputs persist in Drive).

In [3]:
import os, sys, subprocess, time

# 1. GPU verify
try:
    out = subprocess.check_output(["nvidia-smi", "-L"], text=True, timeout=10)
    print(out.strip())
except Exception:
    print("❌ NO GPU. Runtime → Change runtime type → T4 GPU → Save → "
          "Disconnect and delete runtime → reconnect.")
    raise SystemExit(1)

# 2. Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_OUT = '/content/drive/MyDrive/BindCraft_FGFR2/'
os.makedirs(DRIVE_OUT, exist_ok=True)

# 3. JAX GPU sanity
import jax
assert any("gpu" in str(d).lower() or "cuda" in str(d).lower() for d in jax.devices()), \
    f"JAX no GPU: {jax.devices()}"
print(f"JAX: {jax.devices()}")

# 4. Install BindCraft (one-time)
if not os.path.isfile("/content/bindcraft/params/done.txt"):
    print("\n=== Installing BindCraft (~6 min) ===")
    if not os.path.isdir("/content/bindcraft"):
        subprocess.check_call(["git", "clone", "--quiet",
                                "https://github.com/martinpacesa/BindCraft", "/content/bindcraft/"])
        subprocess.run(["chmod", "+x", "/content/bindcraft/functions/dssp"], check=False)
        subprocess.run(["chmod", "+x", "/content/bindcraft/functions/DAlphaBall.gcc"], check=False)
    os.makedirs("/content/bindcraft/params", exist_ok=True)
    # Use wget (more reliable than aria2c with -q quiet mode)
    print("Downloading AF2 params (~5.3 GB, 2-6 min)...")
    subprocess.check_call(["wget", "-q", "--show-progress", "-O",
                            "/content/alphafold_params.tar",
                            "https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar"])
    print("Extracting...")
    subprocess.check_call(["tar", "-xf", "/content/alphafold_params.tar",
                            "-C", "/content/bindcraft/params"])
    open("/content/bindcraft/params/done.txt", "w").close()
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                            "git+https://github.com/sokrypton/ColabDesign.git"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyrosetta_installer"])
    import pyrosetta_installer
    pyrosetta_installer.install_pyrosetta(serialization=True)
    print("✓ BindCraft installed")

# 5. Patch BindCraft's bare exit() bug
gen = "/content/bindcraft/functions/generic_utils.py"
with open(gen) as f:
    src = f.read()
if "raise SystemExit" not in src:
    with open(gen, "w") as f:
        f.write(src.replace("exit()", 'raise SystemExit("No GPU device found")'))

# 6. Repo (the project)
if os.path.isdir("/content/FGFR_binder_hackathon"):
    subprocess.run(["git", "-C", "/content/FGFR_binder_hackathon", "pull", "--quiet"], check=False)
else:
    subprocess.check_call(["git", "clone", "--quiet",
                            "https://github.com/qussai96/FGFR_binder_hackathon.git",
                            "/content/FGFR_binder_hackathon"])

# 7. Configure: 20 designs, v7 hotspots, generous trajectory budget
os.environ["FGFR2_INPUT_PDB"]     = "/content/FGFR_binder_hackathon/FGFR2_1DJS_chainA_clean.pdb"
os.environ["FGFR2_HOTSPOTS"]      = "A283,A281,A346,A317"
os.environ["FGFR1_OFFTARGET_PDB"] = "/content/FGFR_binder_hackathon/FGFR1_1CVS_chainC_clean.pdb"
os.environ["FGFR2_DESIGN_PATH"]   = DRIVE_OUT
os.environ["BINDCRAFT_FOLDER"]    = "/content/bindcraft"

import re
sp = "/content/FGFR_binder_hackathon/fgfr2_campaign/bindcraft/run_bindcraft_fgfr2.py"
with open(sp) as f:
    code = f.read()
code = re.sub(r"NUM_FINAL_DESIGNS\s*=\s*\d+", "NUM_FINAL_DESIGNS = 20", code)
if 'max_trajectories' not in code:
    code = code.replace(
        'advanced_settings["enable_rejection_check"] = True',
        'advanced_settings["enable_rejection_check"] = True\n'
        'advanced_settings["max_trajectories"] = 40'
    )
with open(sp, "w") as f:
    f.write(code)

# 8. RUN
print("\n=== Starting BindCraft — 20 designs target, ~3.5 h on T4 ===")
exec(open(sp).read())

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



❌ NO GPU. Runtime → Change runtime type → T4 GPU → Save → Disconnect and delete runtime → reconnect.
Traceback (most recent call last):
  File "/tmp/ipykernel_50808/2330579185.py", line 5, in <cell line: 0>
    out = subprocess.check_output(["nvidia-smi", "-L"], text=True, timeout=10)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/subprocess.py", line 466, in check_output
    return run(*popenargs, stdout=PIPE, timeout=timeout, check=True,
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "/usr/lib/python3.12/subprocess.py", line 1955, in _execute_child
    raise child_exception_type(errno_num, err_msg, err_filename)
FileNotFou

TypeError: object of type 'NoneType' has no len()

## Cell 3 — Live progress monitor (run any time during Cell 2)

Open a NEW cell and run this any time to check status without interrupting Cell 2.

In [ ]:
import os, glob, pandas as pd

OUT = '/content/drive/MyDrive/BindCraft_FGFR2/'
ts = os.path.join(OUT, 'trajectory_stats.csv')
n_traj = len(pd.read_csv(ts)) if os.path.exists(ts) else 0
n_acc = len(glob.glob(os.path.join(OUT, 'Accepted/*.pdb')))
print(f"Trajectories: {n_traj}   Accepted PDBs: {n_acc} / 20 target")

ranking = os.path.join(OUT, 'bindcraft_frontier_ranking.csv')
if os.path.exists(ranking):
    df = pd.read_csv(ranking).sort_values('frontier_composite', ascending=False)
    cols = [c for c in ['rank','design_name','ipsae_target','ipsae_specificity_gap','frontier_composite'] if c in df.columns]
    print("\nTop 10 by frontier_composite:")
    print(df[cols].head(10).to_string(index=False))

## Cell 4 — Download top 20 outputs as ZIP (run after Cell 2 finishes)

In [ ]:
from google.colab import files
import os, zipfile

OUT = '/content/drive/MyDrive/BindCraft_FGFR2/'
zip_path = '/content/fgfr2_designs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    # Top 5 from Submission folder
    for root, _, fs in os.walk(os.path.join(OUT, 'Submission')):
        for f in fs:
            z.write(os.path.join(root, f), arcname=os.path.join('top_5_pdbs', f))
    # All accepted PDBs
    for root, _, fs in os.walk(os.path.join(OUT, 'Accepted')):
        for f in fs:
            if f.endswith('.pdb'):
                z.write(os.path.join(root, f), arcname=os.path.join('all_accepted_pdbs', f))
    # Master CSVs and FASTA
    for f in ('top_5_for_submission.fasta', 'bindcraft_frontier_ranking.csv',
              'final_design_stats.csv', 'mpnn_design_stats.csv', 'trajectory_stats.csv'):
        p = os.path.join(OUT, f)
        if os.path.exists(p):
            z.write(p, arcname=f)
files.download(zip_path)